# 0.0 Imports

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn import metrics as mt

# 0.1 - Load Datasets

In [6]:
actual_folder = Path.cwd()
print(actual_folder)

main_folder = actual_folder.parent
print(main_folder)

path_folder_dataset = main_folder / 'dataset' / 'classification_datasets'

if path_folder_dataset.exists():
    print(f'Caminho existente: {path_folder_dataset}')
else:
    print(f'Erro: O arquivo não foi encontrado {path_folder_dataset}')

d:\repos\projetos\ml_trials_algorithm\notebooks
d:\repos\projetos\ml_trials_algorithm
Caminho existente: d:\repos\projetos\ml_trials_algorithm\dataset\classification_datasets


In [7]:
#Carrega os datasets da pasta - Treinamento / Validação e Teste

dataset_traning_X = path_folder_dataset / 'a_traninig' / 'X_training.csv'
dataset_traning_y = path_folder_dataset / 'a_traninig' / 'y_training.csv'

dataset_validation_X = path_folder_dataset / 'b_validation' / 'X_validation.csv'
dataset_validation_y = path_folder_dataset / 'b_validation' / 'y_validation.csv'

dataset_test_X = path_folder_dataset / 'c_test' / 'X_test.csv'
dataset_test_y = path_folder_dataset / 'c_test' / 'y_test.csv'

X_train = pd.read_csv(dataset_traning_X)
y_train = pd.read_csv(dataset_traning_y)

X_val = pd.read_csv(dataset_validation_X)
y_val = pd.read_csv(dataset_validation_y)

X_test = pd.read_csv(dataset_test_X)
y_test = pd.read_csv(dataset_test_y)

# Dropa ID do dataset

X_train = X_train.drop(columns=['id'])
X_val = X_val.drop(columns=['id'])
X_test = X_test.drop(columns=['id'])

# 1.0 - Training Algorithm

## Passo 1 — Treino com parâmetros default

In [ ]:
# Instanciar o modelo com parâmetros default
model_def = LogisticRegression(random_state=42)

# Treinar com dados de treino
model_def.fit(X_train, y_train.values.ravel())

# Predições no treino e na validação
yhat_train_def = model_def.predict(X_train)
yhat_val_def   = model_def.predict(X_val)

## Passo 2 — Performance no treino (default)

In [ ]:
# Métricas no conjunto de TREINO com parâmetros default
acuracia_train_def  = mt.accuracy_score(y_train, yhat_train_def)
precision_train_def = mt.precision_score(y_train, yhat_train_def)
recall_train_def    = mt.recall_score(y_train, yhat_train_def)
f1score_train_def   = mt.f1_score(y_train, yhat_train_def)

print("--- Performance no Treino (Default) ---")
print(f"Acurácia:  {acuracia_train_def:.4f}")
print(f"Precisão:  {precision_train_def:.4f}")
print(f"Recall:    {recall_train_def:.4f}")
print(f"F1-Score:  {f1score_train_def:.4f}")

## Passo 3 — Performance na validação (default)

In [ ]:
# Métricas no conjunto de VALIDAÇÃO com parâmetros default
acuracia_val_def  = mt.accuracy_score(y_val, yhat_val_def)
precision_val_def = mt.precision_score(y_val, yhat_val_def)
recall_val_def    = mt.recall_score(y_val, yhat_val_def)
f1score_val_def   = mt.f1_score(y_val, yhat_val_def)

print("--- Performance na Validação (Default) ---")
print(f"Acurácia:  {acuracia_val_def:.4f}")
print(f"Precisão:  {precision_val_def:.4f}")
print(f"Recall:    {recall_val_def:.4f}")
print(f"F1-Score:  {f1score_val_def:.4f}")

## Passo 4 — Ajuste de hiperparâmetros

In [8]:
print("--- Testando Múltiplos Hiperparâmetros Logisic Regression ---")
melhor_c = 1
melhor_solver = 'liblinear'
melhor_max_iter = 100
melhor_f1score_val = 0

# Suas duas listas prontas para o ensaio
list_C = [0.01, 0.1, 1.0, 10.0, 100.0]
list_solver = ['lbfgs', 'liblinear', 'newton-cg', 'sag']
list_max_iter = [100, 500, 1000, 2000]

# LOOP TRIPLO: Cruza C x Solver x Max_Iter
for c_val in list_C:
    for sv in list_solver:
        for mi in list_max_iter:
            
            # Instanciando o modelo com as três variáveis do loop
            model = LogisticRegression(
                penalty = 'l2',
                C=c_val,
                solver=sv,
                max_iter=mi,
                random_state=42
            )
            
            # O Scikit-Learn pode soltar avisos (warnings) se o modelo não convergir.
            # Vamos envelopar o treino para o código rodar limpo.
            try:
                model.fit(X_train, y_train.values.ravel())
                
                yhat_train = model.predict(X_train)
                yhat_val = model.predict(X_val)
                
                f1score_train = mt.f1_score(y_train, yhat_train)
                f1score_val = mt.f1_score(y_val, yhat_val)
                
                print(f"C: {c_val:>6} | Solver: {sv:<10} | Max_Iter: {mi:4d} | F1 Treino: {f1score_train:.4f} | F1 Validação: {f1score_val:.4f}")
                
                # Salva a melhor combinação com base na validação
                if f1score_val > melhor_f1score_val:
                    melhor_f1score_val = f1score_val
                    melhor_C = c_val
                    melhor_solver = sv
                    melhor_max_iter = mi
                    
            except Exception as e:
                # Se algum solver específico falhar por questões matemáticas brutas dos dados, ele pula e avisa
                print(f"Combinação C:{c_val} | Solver:{sv} | Max_Iter:{mi} falhou na convergência.")

print("=" * 70)
print(f"-> VENCEDOR DO ENSAIO DA REGRESSÃO LOGÍSTICA:")
print(f"Melhor C: {melhor_C}")
print(f"Melhor Solver: {melhor_solver}")
print(f"Melhor Max_Iter: {melhor_max_iter}")
print(f"Maior F1-Score na Validação: {melhor_f1score_val:.4f}\n")

--- Testando Múltiplos Hiperparâmetros Logisic Regression ---
C:   0.01 | Solver: lbfgs      | Max_Iter:  100 | F1 Treino: 0.8520 | F1 Validação: 0.8499
C:   0.01 | Solver: lbfgs      | Max_Iter:  500 | F1 Treino: 0.8520 | F1 Validação: 0.8499
C:   0.01 | Solver: lbfgs      | Max_Iter: 1000 | F1 Treino: 0.8520 | F1 Validação: 0.8499
C:   0.01 | Solver: lbfgs      | Max_Iter: 2000 | F1 Treino: 0.8520 | F1 Validação: 0.8499
C:   0.01 | Solver: liblinear  | Max_Iter:  100 | F1 Treino: 0.8512 | F1 Validação: 0.8493
C:   0.01 | Solver: liblinear  | Max_Iter:  500 | F1 Treino: 0.8512 | F1 Validação: 0.8493
C:   0.01 | Solver: liblinear  | Max_Iter: 1000 | F1 Treino: 0.8512 | F1 Validação: 0.8493
C:   0.01 | Solver: liblinear  | Max_Iter: 2000 | F1 Treino: 0.8512 | F1 Validação: 0.8493
C:   0.01 | Solver: newton-cg  | Max_Iter:  100 | F1 Treino: 0.8521 | F1 Validação: 0.8496
C:   0.01 | Solver: newton-cg  | Max_Iter:  500 | F1 Treino: 0.8521 | F1 Validação: 0.8496
C:   0.01 | Solver: newton-c

c:\Users\Guilherme Grandim\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


C:   10.0 | Solver: sag        | Max_Iter:  100 | F1 Treino: 0.8531 | F1 Validação: 0.8518
C:   10.0 | Solver: sag        | Max_Iter:  500 | F1 Treino: 0.8531 | F1 Validação: 0.8518
C:   10.0 | Solver: sag        | Max_Iter: 1000 | F1 Treino: 0.8531 | F1 Validação: 0.8518
C:   10.0 | Solver: sag        | Max_Iter: 2000 | F1 Treino: 0.8531 | F1 Validação: 0.8518
C:  100.0 | Solver: lbfgs      | Max_Iter:  100 | F1 Treino: 0.8532 | F1 Validação: 0.8519
C:  100.0 | Solver: lbfgs      | Max_Iter:  500 | F1 Treino: 0.8532 | F1 Validação: 0.8519
C:  100.0 | Solver: lbfgs      | Max_Iter: 1000 | F1 Treino: 0.8532 | F1 Validação: 0.8519
C:  100.0 | Solver: lbfgs      | Max_Iter: 2000 | F1 Treino: 0.8532 | F1 Validação: 0.8519
C:  100.0 | Solver: liblinear  | Max_Iter:  100 | F1 Treino: 0.8530 | F1 Validação: 0.8519
C:  100.0 | Solver: liblinear  | Max_Iter:  500 | F1 Treino: 0.8530 | F1 Validação: 0.8519
C:  100.0 | Solver: liblinear  | Max_Iter: 1000 | F1 Treino: 0.8530 | F1 Validação: 0.8519

c:\Users\Guilherme Grandim\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


C:  100.0 | Solver: sag        | Max_Iter:  100 | F1 Treino: 0.8531 | F1 Validação: 0.8519


c:\Users\Guilherme Grandim\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


C:  100.0 | Solver: sag        | Max_Iter:  500 | F1 Treino: 0.8531 | F1 Validação: 0.8519
C:  100.0 | Solver: sag        | Max_Iter: 1000 | F1 Treino: 0.8531 | F1 Validação: 0.8519
C:  100.0 | Solver: sag        | Max_Iter: 2000 | F1 Treino: 0.8531 | F1 Validação: 0.8519
-> VENCEDOR DO ENSAIO DA REGRESSÃO LOGÍSTICA:
Melhor C: 1.0
Melhor Solver: lbfgs
Melhor Max_Iter: 100
Maior F1-Score na Validação: 0.8519



## Passo 5 — União treino + validação

In [ ]:
# Unir treino + validação para formar o conjunto final de treinamento
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

print(f"X_train_final shape: {X_train_final.shape}")
print(f"y_train_final shape: {y_train_final.shape}")

## Passo 6 — Retreino com melhores parâmetros

In [9]:
c_final = 1.0
solver_final = 'lbfgs'
max_iter_final = 100

# Passando os dois parâmetros variáveis ao mesmo tempo
model = LogisticRegression( C=c_final,
                            solver=solver_final,
                            max_iter=max_iter_final,
                            random_state=42 )
        
# Training
model.fit(X_train, y_train.values.ravel())
        
# Predict
yhat_train = model.predict(X_train)
yhat_val = model.predict(X_val)
        
# Métricas de Avaliação do Modelo (Métrica chave do laboratório)
acuracia_train = mt.accuracy_score(y_train, yhat_train)
acuracia_val = mt.accuracy_score(y_val, yhat_val)

precision_train = mt.precision_score(y_train, yhat_train)
precision_val = mt.precision_score(y_val, yhat_val)

recall_train = mt.recall_score(y_train, yhat_train)
recall_val = mt.recall_score(y_val, yhat_val)

f1score_train = mt.f1_score(y_train, yhat_train)
f1score_val = mt.f1_score(y_val, yhat_val)


print(f"C: {c_final:>6} | Solver: {solver_final:<10} | Max_Iter: {max_iter_final:4d} | Acuracia Treino: {acuracia_train:.4f} | Acuracia Validação: {acuracia_val:.4f}")
print(f"C: {c_final:>6} | Solver: {solver_final:<10} | Max_Iter: {max_iter_final:4d} | Precision Treino: {precision_train:.4f} | Precision Validação: {precision_val:.4f}")
print(f"C: {c_final:>6} | Solver: {solver_final:<10} | Max_Iter: {max_iter_final:4d} | Recall Treino: {recall_train:.4f} | Recall Validação: {recall_val:.4f}")
print(f"C: {c_final:>6} | Solver: {solver_final:<10} | Max_Iter: {max_iter_final:4d} | F1 Treino: {f1score_train:.4f} | F1 Validação: {f1score_val:.4f}")

C:    1.0 | Solver: lbfgs      | Max_Iter:  100 | Acuracia Treino: 0.8753 | Acuracia Validação: 0.8742
C:    1.0 | Solver: lbfgs      | Max_Iter:  100 | Precision Treino: 0.8707 | Precision Validação: 0.8692
C:    1.0 | Solver: lbfgs      | Max_Iter:  100 | Recall Treino: 0.8364 | Recall Validação: 0.8353
C:    1.0 | Solver: lbfgs      | Max_Iter:  100 | F1 Treino: 0.8532 | F1 Validação: 0.8519


## Passo 7 — Performance no teste

In [10]:
# ==============================================================================
# JUNTAR AS BASES E EXECUTAR O TESTE FINAL
# ==============================================================================
print("--- Executando o Teste Final ---")

melhor_c_test = melhor_C
melhor_solver_test = melhor_solver
melhor_max_iter_test = melhor_max_iter

# Juntando treino e validação para dar mais volume de dados ao KNN
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

# Treinando o modelo definitivo com o melhor K encontrado
model_final = LogisticRegression(   C=melhor_c_test,
                                    solver=melhor_solver_test,
                                    max_iter=melhor_max_iter_test,
                                    random_state=42)
model_final.fit(X_train_final, y_train_final.values.ravel())

# Previsão final usando o dataset de teste (que ficou isolado o tempo todo)
yhat_test = model_final.predict(X_test)

#Performance Test
accuracia_test = mt.accuracy_score(y_test, yhat_test)
precisao_test = mt.precision_score(y_test, yhat_test)
recall_test = mt.recall_score(y_test, yhat_test)
f1score_test = mt.f1_score(y_test, yhat_test)

# Métrica final de produção
print(f"Acurácia Teste: {accuracia_test:.4f}")
print(f"Precisão Teste: {precisao_test:.4f}")
print(f"Recall Teste:   {recall_test:.4f}")
print(f"F1-Score Teste: {f1score_test:.4f}")

--- Executando o Teste Final ---
Acurácia Teste: 0.8711
Precisão Teste: 0.8685
Recall Teste:   0.8324
F1-Score Teste: 0.8501


## Passo 9 — Quadro Comparativo — Diagnóstico Completo

In [ ]:
data_comparativo = {
    'Conjunto': ['Treino (Default)', 'Validação (Default)', 'Treino (Tunado)', 'Validação (Tunado)', 'Teste (Final)'],
    'Acurácia': [acuracia_train_def, acuracia_val_def, '-', '-', accuracia_test],
    'Precisão': [precision_train_def, precision_val_def, '-', '-', precisao_test],
    'Recall':   [recall_train_def, recall_val_def, '-', '-', recall_test],
    'F1-Score': [f1score_train_def, f1score_val_def, '-', '-', f1score_test],
}
df_comparativo = pd.DataFrame(data_comparativo)
print("\n--- Quadro Comparativo — Diagnóstico Completo ---")
print(df_comparativo.to_string(index=False))